In [1]:
"""
Knowledge Distillation — Script 6: Cross-Architecture Distillation
===================================================================
Teacher : ResNet-50 (modified for CIFAR-10, pre-trained baseline)
Student : MobileNetV2 (modified for CIFAR-10)
Dataset : CIFAR-10
Method  : Cross-Architecture KD — CNN→CNN with incompatible internal spaces

WHAT IS CROSS-ARCHITECTURE KD?
================================
When teacher and student use different architectures, their internal feature
spaces are structurally incompatible — you can't directly subtract ResNet feature
maps from MobileNet feature maps. We need architecture-agnostic similarity measures.

Three approaches implemented:

1. Response-Based (output-only, architecture-agnostic baseline)
   Plain KL divergence on soft logits — no internal access needed.
   This is the minimum viable cross-arch KD.
   L = alpha*CE + (1-alpha)*tau^2*KL(p_teacher || p_student)

2. CKA-Guided Projection Distillation
   Uses a shared projection head to map BOTH teacher and student features
   into a common latent space R^d, then applies MSE there.
   L = CE + beta * ||g_T(F_T) - g_S(F_S)||_F^2
   where g_T, g_S are lightweight MLP projectors (512→256→d).
   Centred Kernel Alignment (CKA) is logged at each epoch to measure
   how well representations are being aligned (not used in the loss directly).

3. Contrastive Alignment (InfoNCE-style)
   Treats each teacher-student pair as a positive pair and all other
   student embeddings in the batch as negatives. Maximises mutual information:
   L = CE + gamma * L_InfoNCE
   L_InfoNCE = -log[ exp(sim(z_T, z_S^+)/temp) / Σ_k exp(sim(z_T, z_S^k)/temp) ]
   where sim is cosine similarity and z are projected embeddings.

CKA computation (logged but not used in gradient):
  CKA(K, L) = HSIC(K,L) / sqrt(HSIC(K,K) * HSIC(L,L))
  HSIC(K,L) = tr(KHLH) / (n-1)^2,  H = I - (1/n)*11^T

Sweeps:
  A — method      : [response, projection, contrastive]   (default params)
  B — projection_d: [64, 128, 256]                        (projection only)
  C — alpha/beta/gamma: varies per best method

Output : __6__KD_cross_architecture.json
"""

import os, sys, json, time, copy, tempfile, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

warnings.filterwarnings("ignore")

# ── Device ─────────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[init] PyTorch {torch.__version__}", flush=True)
print(f"[init] Device  : {DEVICE}", flush=True)
if DEVICE.type == "cuda":
    print(f"[init] GPU     : {torch.cuda.get_device_name(0)}", flush=True)

# ── Config ─────────────────────────────────────────────────────────────────────
TEACHER_MODEL_PATH    = "../__2__baseline_resnet50_cifar10.pth"
BASELINE_METRICS_PATH = "../__2__baseline_metrics.json"
OUTPUT_JSON           = "__6__KD_cross_architecture.json"

BATCH_SIZE  = 128 if DEVICE.type == "cuda" else 64
NUM_WORKERS = 0
PIN_MEMORY  = False
NUM_CLASSES = 10
KD_EPOCHS   = 10
USE_AMP     = (DEVICE.type == "cuda")

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

print(f"[init] AMP: {USE_AMP}", flush=True)


# ── Model builders ─────────────────────────────────────────────────────────────
def build_resnet50_cifar(num_classes: int = 10) -> nn.Module:
    """Pre-trained teacher."""
    model = models.resnet50(weights=None)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(model.fc.in_features, num_classes)
    return model

def build_mobilenetv2_cifar(num_classes: int = 10) -> nn.Module:
    """
    MobileNetV2 modified for CIFAR-10 (32×32 images):
    - Replace first conv stride=2 with stride=1 (images are too small for stride-2)
    - Replace classifier to output num_classes
    """
    model = models.mobilenet_v2(weights=None)
    # Fix stride for small images
    model.features[0][0] = nn.Conv2d(
        3, 32, kernel_size=3, stride=1, padding=1, bias=False
    )
    # Fix classifier
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model

def load_teacher(path: str) -> nn.Module:
    print(f"[load] Loading teacher from {path} ...", flush=True)
    model = build_resnet50_cifar(NUM_CLASSES)
    model.load_state_dict(torch.load(path, map_location="cpu"))
    model.eval().to(DEVICE)
    print("[load] Teacher OK", flush=True)
    return model


# ── Embedding extractor (avgpool output) ───────────────────────────────────────
class EmbeddingExtractor(nn.Module):
    """Hooks avgpool of ResNet-50 to get 2048-d embedding."""
    def __init__(self, model: nn.Module):
        super().__init__()
        self.model  = model
        self._embed = None
        self._hook  = model.avgpool.register_forward_hook(
            lambda m, i, o: setattr(self, "_embed", o.view(o.size(0), -1))
        )

    def forward(self, x):
        logits = self.model(x)
        return logits, self._embed

    def remove_hook(self): self._hook.remove()


class MobileNetEmbeddingExtractor(nn.Module):
    """Hooks adaptive_avg_pool of MobileNetV2 to get 1280-d embedding."""
    def __init__(self, model: nn.Module):
        super().__init__()
        self.model  = model
        self._embed = None
        # MobileNetV2 last feature → adaptive_avg_pool2d → flatten
        # We hook the last features block
        self._hook  = model.features.register_forward_hook(
            lambda m, i, o: setattr(
                self, "_embed",
                F.adaptive_avg_pool2d(o, 1).view(o.size(0), -1)
            )
        )

    def forward(self, x):
        logits = self.model(x)
        return logits, self._embed

    def remove_hook(self): self._hook.remove()


# ── Projection head ────────────────────────────────────────────────────────────
class ProjectionHead(nn.Module):
    """
    MLP projector: in_dim → hidden → proj_dim (L2-normalized output).
    Used to map both teacher and student into a shared latent space.
    """
    def __init__(self, in_dim: int, proj_dim: int = 128, hidden: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, proj_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.normalize(self.net(x), p=2, dim=1)


# ── CKA metric (logged, not used in loss) ──────────────────────────────────────
def linear_cka(X: torch.Tensor, Y: torch.Tensor) -> float:
    """
    Compute linear CKA between representations X (n×p) and Y (n×q).
    CKA = HSIC(XX^T, YY^T) / sqrt(HSIC(XX^T, XX^T) * HSIC(YY^T, YY^T))
    """
    def centering(K: torch.Tensor) -> torch.Tensor:
        n = K.size(0)
        H = torch.eye(n, device=K.device) - (1.0 / n) * torch.ones(n, n, device=K.device)
        return H @ K @ H

    def hsic(Kc: torch.Tensor, Lc: torch.Tensor) -> float:
        n = Kc.size(0)
        return float((Kc * Lc).sum() / ((n - 1) ** 2))

    with torch.no_grad():
        X = X - X.mean(0, keepdim=True)
        Y = Y - Y.mean(0, keepdim=True)
        Kx = X @ X.t()
        Ky = Y @ Y.t()
        Kxc = centering(Kx)
        Kyc = centering(Ky)
        num   = hsic(Kxc, Kyc)
        denom = np.sqrt(max(hsic(Kxc, Kxc), 1e-10) * max(hsic(Kyc, Kyc), 1e-10))
        return float(num / denom)


# ── InfoNCE loss ───────────────────────────────────────────────────────────────
def infonce_loss(z_t: torch.Tensor, z_s: torch.Tensor, temp: float = 0.07) -> torch.Tensor:
    """
    Contrastive alignment loss.
    Treats (z_t[i], z_s[i]) as positive pair, all other z_s[j≠i] as negatives.
    z_t, z_s must be L2-normalized projections of shape (B, d).
    """
    B = z_t.size(0)
    # Similarity matrix (B, B)
    sim = z_t @ z_s.t() / temp   # (B, B)
    # Labels: diagonal is the positive pair for each query
    labels = torch.arange(B, device=z_t.device)
    # Cross-entropy loss over similarity rows
    return F.cross_entropy(sim, labels)


# ── Data ────────────────────────────────────────────────────────────────────────
def get_train_loader():
    transform = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = torchvision.datasets.CIFAR10(root="../data", train=True,
                                       download=True, transform=transform)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

def get_test_loader():
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = torchvision.datasets.CIFAR10(root="../data", train=False,
                                       download=True, transform=transform)
    return DataLoader(ds, batch_size=512, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

# ── Helpers ─────────────────────────────────────────────────────────────────────
def model_size_mb(model: nn.Module) -> float:
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        torch.save(model.state_dict(), tmp)
        return os.path.getsize(tmp) / 1024 ** 2
    finally:
        os.remove(tmp)

def param_count(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())

def sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader) -> dict:
    model.eval()
    preds, labels = [], []
    for inputs, lbls in loader:
        inputs = inputs.to(DEVICE, non_blocking=True)
        out = model(inputs)
        logits = out[0] if isinstance(out, tuple) else out
        preds.extend(logits.argmax(1).cpu().numpy())
        labels.extend(lbls.numpy())
    y_pred, y_true = np.array(preds), np.array(labels)
    return {
        "accuracy" : float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall"   : float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1"       : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }

def measure_inference_ms(model: nn.Module, runs: int = 30) -> float:
    m = copy.deepcopy(model).to(DEVICE).eval()
    dummy = torch.randn(128, 3, 32, 32, device=DEVICE)
    with torch.no_grad():
        for _ in range(5): m(dummy)
        sync()
        t0 = time.perf_counter()
        for _ in range(runs): m(dummy)
        sync()
    return (time.perf_counter() - t0) / runs * 1000


# ══════════════════════════════════════════════════════════════════════════════
# Core cross-arch KD loop
# ══════════════════════════════════════════════════════════════════════════════
def run_cross_arch_kd(
    teacher_raw: nn.Module,
    train_loader: DataLoader,
    test_loader: DataLoader,
    baseline_acc: float,
    method: str      = "response",   # "response" | "projection" | "contrastive"
    tau: float       = 4.0,
    alpha: float     = 0.1,          # CE weight for response KD
    beta: float      = 1.0,          # projection loss weight
    gamma: float     = 0.5,          # contrastive loss weight
    proj_dim: int    = 128,          # shared projection dimension
    proj_temp: float = 0.07,         # InfoNCE temperature
    lr: float        = 0.01,
    n_epochs: int    = KD_EPOCHS,
    sweep_name: str  = "method",
) -> dict:
    tag = (f"{method}/tau={tau}/alpha={alpha}"
           if method == "response"
           else f"{method}/proj_dim={proj_dim}/beta={beta}/gamma={gamma}")
    print(f"\n  ┌─ [{tag}]", flush=True)

    try:
        student_raw = build_mobilenetv2_cifar(NUM_CLASSES).to(DEVICE)

        # Teacher embedding dim: 2048 (ResNet-50 avgpool)
        # Student embedding dim: 1280 (MobileNetV2 last features)
        T_DIM = 2048
        S_DIM = 1280

        # Wrappers for embedding extraction (needed for projection + contrastive)
        teacher_extractor = None
        student_extractor = None
        proj_t = proj_s = None

        if method in ("projection", "contrastive"):
            teacher_extractor = EmbeddingExtractor(teacher_raw).to(DEVICE)
            student_extractor = MobileNetEmbeddingExtractor(student_raw).to(DEVICE)
            proj_t = ProjectionHead(T_DIM, proj_dim).to(DEVICE)
            proj_s = ProjectionHead(S_DIM, proj_dim).to(DEVICE)
            print(f"      [proj] T_DIM={T_DIM}→{proj_dim}  S_DIM={S_DIM}→{proj_dim}", flush=True)

        # Parameters
        params = list(student_raw.parameters())
        if proj_s is not None:
            params += list(proj_s.parameters())
        if proj_t is not None:
            params += list(proj_t.parameters())

        optimizer = torch.optim.SGD(params, lr=lr,
                                    momentum=0.9, weight_decay=1e-4, nesterov=True)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
        scaler    = torch.amp.GradScaler(enabled=USE_AMP)

        if DEVICE.type == "cuda":
            torch.cuda.reset_peak_memory_stats()

        epoch_train_acc  = []
        epoch_align_loss = []
        epoch_cka        = []
        t_start = time.perf_counter()

        teacher_raw.eval()

        for epoch in range(n_epochs):
            student_raw.train()
            if proj_s: proj_s.train()
            if proj_t: proj_t.train()

            correct = total = 0
            align_sum = 0.0
            t_ep = time.perf_counter()
            cka_batch_vals = []

            for batch_idx, (inputs, targets) in enumerate(train_loader):
                inputs  = inputs.to(DEVICE, non_blocking=True)
                targets = targets.to(DEVICE, non_blocking=True)

                optimizer.zero_grad(set_to_none=True)

                with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                    with torch.no_grad():
                        if method == "response":
                            t_logits = teacher_raw(inputs)
                        else:
                            t_logits, t_emb = teacher_extractor(inputs)

                    if method == "response":
                        s_logits = student_raw(inputs)
                        ce_loss  = F.cross_entropy(s_logits, targets)
                        p_t      = F.softmax(teacher_raw(inputs).detach() / tau, dim=1)
                        # Reuse t_logits
                        p_t      = F.softmax(t_logits / tau, dim=1)
                        p_s_log  = F.log_softmax(s_logits / tau, dim=1)
                        kl       = F.kl_div(p_s_log, p_t, reduction="batchmean") * (tau**2)
                        loss     = alpha * ce_loss + (1.0 - alpha) * kl
                        align_loss = kl

                    elif method == "projection":
                        s_logits, s_emb = student_extractor(inputs)
                        ce_loss  = F.cross_entropy(s_logits, targets)
                        z_t = proj_t(t_emb.detach())
                        z_s = proj_s(s_emb)
                        proj_loss = F.mse_loss(z_s, z_t.detach())
                        loss = ce_loss + beta * proj_loss
                        align_loss = proj_loss

                        # Log CKA every 50 batches (lightweight)
                        if (batch_idx + 1) % 50 == 0:
                            with torch.no_grad():
                                cka_val = linear_cka(
                                    t_emb[:32].float(), s_emb[:32].float()
                                )
                                cka_batch_vals.append(cka_val)

                    else:  # contrastive
                        s_logits, s_emb = student_extractor(inputs)
                        ce_loss  = F.cross_entropy(s_logits, targets)
                        z_t = proj_t(t_emb.detach())
                        z_s = proj_s(s_emb)
                        cont_loss = infonce_loss(z_t, z_s, temp=proj_temp)
                        loss = ce_loss + gamma * cont_loss
                        align_loss = cont_loss

                        if (batch_idx + 1) % 50 == 0:
                            with torch.no_grad():
                                cka_val = linear_cka(
                                    t_emb[:32].float(), s_emb[:32].float()
                                )
                                cka_batch_vals.append(cka_val)

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

                correct += (s_logits.detach().argmax(1) == targets).sum().item()
                total   += targets.size(0)
                align_sum += align_loss.item() * targets.size(0)

                if (batch_idx + 1) % 100 == 0:
                    elapsed = time.perf_counter() - t_ep
                    done = batch_idx + 1
                    eta  = elapsed / done * (len(train_loader) - done)
                    print(f"      [epoch {epoch+1}/{n_epochs}] "
                          f"batch {done}/{len(train_loader)}  "
                          f"acc={correct/total:.4f}  "
                          f"align={align_sum/total:.4f}  ETA={eta:.0f}s", flush=True)

            scheduler.step()
            acc = correct / total
            epoch_train_acc.append(round(acc, 6))
            epoch_align_loss.append(round(align_sum / total, 6))
            if cka_batch_vals:
                epoch_cka.append(round(float(np.mean(cka_batch_vals)), 4))

            sync()
            ep_time = time.perf_counter() - t_ep
            remaining = n_epochs - (epoch + 1)
            cka_str = f"  CKA={epoch_cka[-1]:.3f}" if epoch_cka else ""
            print(f"      [epoch {epoch+1}/{n_epochs}] DONE  "
                  f"acc={acc:.4f}  align={align_sum/total:.4f}{cka_str}  "
                  f"time={ep_time:.1f}s  ETA_remaining={ep_time*remaining:.0f}s", flush=True)

        sync()
        train_time_s = time.perf_counter() - t_start
        peak_gpu_mb  = (round(torch.cuda.max_memory_allocated() / 1024**2, 2)
                        if DEVICE.type == "cuda" else None)

        # Clean up hooks
        if teacher_extractor: teacher_extractor.remove_hook()
        if student_extractor: student_extractor.remove_hook()

        print("      [eval] Evaluating student ...", flush=True)
        metrics      = evaluate(student_raw, test_loader)
        inference_ms = measure_inference_ms(student_raw)
        size_mb      = model_size_mb(student_raw)
        acc_drop     = baseline_acc - metrics["accuracy"]

        print(f"  └─ Acc={metrics['accuracy']:.4f}  Drop={acc_drop:+.4f}  "
              f"Size={size_mb:.2f}MB  Inf={inference_ms:.1f}ms  "
              f"Train={train_time_s:.1f}s", flush=True)

        return {
            "sweep"             : sweep_name,
            "method"            : method,
            "temperature"       : tau,
            "alpha"             : alpha,
            "beta"              : beta,
            "gamma"             : gamma,
            "proj_dim"          : proj_dim,
            "proj_temp"         : proj_temp,
            "lr"                : lr,
            "lr_schedule"       : "cosine",
            "epochs"            : n_epochs,
            "train_device"      : str(DEVICE),
            "use_amp"           : USE_AMP,
            "train_time_s"      : round(train_time_s, 2),
            "epoch_train_acc"   : epoch_train_acc,
            "epoch_align_loss"  : epoch_align_loss,
            "epoch_cka"         : epoch_cka,
            "accuracy"          : round(metrics["accuracy"],  6),
            "precision"         : round(metrics["precision"], 6),
            "recall"            : round(metrics["recall"],    6),
            "f1"                : round(metrics["f1"],        6),
            "accuracy_drop"     : round(acc_drop, 6),
            "size_mb"           : round(size_mb, 4),
            "inference_ms"      : round(inference_ms, 4),
            "peak_gpu_memory_mb": peak_gpu_mb,
            "student_params"    : param_count(student_raw),
            "description"       : (
                f"Cross-Arch KD ResNet50→MobileNetV2 ({method}, "
                f"proj_dim={proj_dim}, tau={tau}, epochs={n_epochs})"
            ),
            "status": "ok",
        }

    except Exception as e:
        import traceback
        print(f"  └─ FAILED: {e}", flush=True)
        traceback.print_exc()
        return {
            "sweep": sweep_name, "method": method,
            "status": "failed", "error": str(e),
            "accuracy": None, "accuracy_drop": None,
        }


# ── Main ────────────────────────────────────────────────────────────────────────
def main():
    print(f"\n{'='*65}")
    print("  Cross-Architecture KD  (ResNet-50 → MobileNetV2)")
    print(f"  Device: {DEVICE}  |  Epochs: {KD_EPOCHS}  |  Batch: {BATCH_SIZE}")
    print(f"{'='*65}\n")

    with open(BASELINE_METRICS_PATH) as f:
        baseline = json.load(f)
    baseline_acc = baseline["accuracy"]
    print(f"  Baseline teacher acc: {baseline_acc:.4f}\n", flush=True)

    teacher = load_teacher(TEACHER_MODEL_PATH)
    student_ref = build_mobilenetv2_cifar(NUM_CLASSES)
    teacher_size_mb = model_size_mb(teacher)
    student_size_mb = model_size_mb(student_ref)
    teacher_inf_ms  = measure_inference_ms(teacher)
    student_inf_ms  = measure_inference_ms(student_ref)

    print(f"  Teacher (ResNet-50) — Size: {teacher_size_mb:.2f} MB  "
          f"Params: {param_count(teacher):,}  Inf: {teacher_inf_ms:.1f}ms")
    print(f"  Student (MobileNetV2) — Size: {student_size_mb:.2f} MB  "
          f"Params: {param_count(student_ref):,}  Inf: {student_inf_ms:.1f}ms\n")

    train_loader = get_train_loader()
    test_loader  = get_test_loader()

    results = []

    # ── Sweep A: Method ───────────────────────────────────────────────────────
    print("  ── Sweep A: Method ───────────────────────────────────────────────", flush=True)
    sweep_a = []
    for method in ["response", "projection", "contrastive"]:
        row = run_cross_arch_kd(
            teacher, train_loader, test_loader, baseline_acc,
            method=method, tau=4.0, alpha=0.1, beta=1.0, gamma=0.5,
            proj_dim=128, proj_temp=0.07,
            lr=0.01, n_epochs=KD_EPOCHS, sweep_name="A_method",
        )
        results.append(row)
        sweep_a.append(row)

    valid_a = [r for r in sweep_a if r.get("accuracy") is not None]
    if not valid_a:
        print("  ✗ Sweep A failed.", flush=True)
        return
    best_method = max(valid_a, key=lambda r: r["accuracy"])["method"]
    print(f"\n  Best method: {best_method}", flush=True)

    # ── Sweep B: Projection dim (if projection/contrastive won) ───────────────
    if best_method in ("projection", "contrastive"):
        print(f"\n  ── Sweep B: Projection dim  (method={best_method}) ───────────────", flush=True)
        for proj_dim in [64, 128, 256]:
            row = run_cross_arch_kd(
                teacher, train_loader, test_loader, baseline_acc,
                method=best_method, tau=4.0, alpha=0.1, beta=1.0, gamma=0.5,
                proj_dim=proj_dim, proj_temp=0.07,
                lr=0.01, n_epochs=KD_EPOCHS, sweep_name="B_proj_dim",
            )
            results.append(row)

        valid_b = [r for r in results
                   if r.get("sweep") == "B_proj_dim" and r.get("accuracy") is not None]
        best_proj_dim = max(valid_b, key=lambda r: r["accuracy"])["proj_dim"] if valid_b else 128
    else:
        best_proj_dim = 128

    # ── Sweep C: Loss weight (alpha for response, beta/gamma for others) ──────
    print(f"\n  ── Sweep C: Loss weight  (method={best_method}) ──────────────────", flush=True)
    if best_method == "response":
        for alpha in [0.05, 0.1, 0.3, 0.5]:
            row = run_cross_arch_kd(
                teacher, train_loader, test_loader, baseline_acc,
                method="response", tau=4.0, alpha=alpha,
                proj_dim=best_proj_dim, lr=0.01, n_epochs=KD_EPOCHS,
                sweep_name="C_loss_weight",
            )
            results.append(row)
    elif best_method == "projection":
        for beta in [0.1, 0.5, 1.0, 5.0]:
            row = run_cross_arch_kd(
                teacher, train_loader, test_loader, baseline_acc,
                method="projection", tau=4.0, beta=beta,
                proj_dim=best_proj_dim, lr=0.01, n_epochs=KD_EPOCHS,
                sweep_name="C_loss_weight",
            )
            results.append(row)
    else:  # contrastive
        for gamma in [0.1, 0.5, 1.0, 2.0]:
            row = run_cross_arch_kd(
                teacher, train_loader, test_loader, baseline_acc,
                method="contrastive", tau=4.0, gamma=gamma,
                proj_dim=best_proj_dim, lr=0.01, n_epochs=KD_EPOCHS,
                sweep_name="C_loss_weight",
            )
            results.append(row)

    # ── Best overall ──────────────────────────────────────────────────────────
    valid = [r for r in results if r.get("accuracy") is not None]
    if not valid:
        print("  ✗ No valid results.", flush=True)
        return
    best = max(valid, key=lambda r: r["accuracy"])

    report = {
        "method"        : "cross_architecture_kd",
        "description"   : "Cross-Architecture KD: ResNet-50 → MobileNetV2 (CIFAR-10)",
        "teacher_arch"  : "ResNet-50 (CIFAR-10 modified)",
        "student_arch"  : "MobileNetV2 (CIFAR-10 modified)",
        "dataset"       : "CIFAR-10",
        "train_device"  : str(DEVICE),
        "use_amp"       : USE_AMP,
        "kd_epochs"     : KD_EPOCHS,
        "baseline"      : baseline,
        "teacher_info"  : {
            "size_mb"     : round(teacher_size_mb, 4),
            "inference_ms": round(teacher_inf_ms, 4),
            "params"      : param_count(teacher),
        },
        "student_info"  : {
            "size_mb"     : round(student_size_mb, 4),
            "inference_ms": round(student_inf_ms, 4),
            "params"      : param_count(student_ref),
        },
        "best_config"   : {
            "method"       : best["method"],
            "proj_dim"     : best.get("proj_dim"),
            "accuracy"     : best["accuracy"],
            "accuracy_drop": best["accuracy_drop"],
            "size_mb"      : best["size_mb"],
            "inference_ms" : best["inference_ms"],
        },
        "sweeps"        : {
            "A_method"     : "response vs projection vs contrastive",
            "B_proj_dim"   : "proj_dim in [64, 128, 256]  (best method)",
            "C_loss_weight": "loss weight sweep  (best method + proj_dim)",
        },
        "results"       : results,
    }

    with open(OUTPUT_JSON, "w") as f:
        json.dump(report, f, indent=2)

    print(f"\n  ✓ Saved → {OUTPUT_JSON}")
    print(f"  Best: method={best['method']} | Acc={best['accuracy']:.4f} | "
          f"Drop={best['accuracy_drop']:+.4f} | "
          f"Size={best['size_mb']:.2f}MB (vs teacher {teacher_size_mb:.2f}MB)")
    print(f"\n  Teacher params : {param_count(teacher):,}")
    print(f"  Student params : {param_count(student_ref):,}  "
          f"({param_count(student_ref)/param_count(teacher)*100:.1f}% of teacher)")


if __name__ == "__main__":
    main()

[init] PyTorch 2.12.0.dev20260324+cu128
[init] Device  : cuda
[init] GPU     : NVIDIA GeForce RTX 5070 Laptop GPU
[init] AMP: True

  Cross-Architecture KD  (ResNet-50 → MobileNetV2)
  Device: cuda  |  Epochs: 10  |  Batch: 128

  Baseline teacher acc: 0.9320

[load] Loading teacher from ../__2__baseline_resnet50_cifar10.pth ...
[load] Teacher OK
  Teacher (ResNet-50) — Size: 90.03 MB  Params: 23,520,842  Inf: 307.8ms
  Student (MobileNetV2) — Size: 8.76 MB  Params: 2,236,682  Inf: 17.3ms

  ── Sweep A: Method ───────────────────────────────────────────────

  ┌─ [response/tau=4.0/alpha=0.1]
      [epoch 1/10] batch 100/391  acc=0.1850  align=1.5352  ETA=191s
      [epoch 1/10] batch 200/391  acc=0.2509  align=1.4480  ETA=116s
      [epoch 1/10] batch 300/391  acc=0.2952  align=1.3876  ETA=55s
      [epoch 1/10] DONE  acc=0.3223  align=1.3486  time=241.1s  ETA_remaining=2170s
      [epoch 2/10] batch 100/391  acc=0.4509  align=1.1562  ETA=145s
      [epoch 2/10] batch 200/391  acc=0.45